# Create the train datastracture for 3 edges

In [2]:
import pandas as pd

# Specify the features to keep
selected_features = [
    'Src IP', 'Dst IP', 'Timestamp', 'Label', 
    'Src Port', 'Dst Port', 'Fwd Header Len', 'Init Bwd Win Byts', 
    'Fwd Seg Size Avg', 'Fwd Pkt Len Mean', 'Init Fwd Win Byts', 
    'Fwd Pkt Len Max', 'TotLen Fwd Pkts', 'Bwd Pkt Len Mean', 
    'Idle Min', 'Bwd Header Len', 'Pkt Len Var', 'Subflow Fwd Byts', 
    'TotLen Bwd Pkts', 'Idle Max', 'Fwd Seg Size Min', 'Idle Mean', 
    'Pkt Len Max', 'Bwd Pkt Len Std', 'Bwd Pkt Len Max', 
    'Protocol', 'Pkt Len Mean', 'Down/Up Ratio'
]

# Load your dataset
data = pd.read_csv('train_data.csv')  # Replace 'your_data.csv' with your actual file name

# Keep only the specified features
filtered_data = data[selected_features]

# Convert 'Timestamp' to datetime
filtered_data['Timestamp'] = pd.to_datetime(filtered_data['Timestamp'])

# Order the data by 'Timestamp'
filtered_data = filtered_data.sort_values(by='Timestamp')

# Save the temporally ordered data to a new file
filtered_data.to_csv('filtered_train_3edge.csv', index=False)

print("Filtered and temporally ordered data saved to 'filtered_train_3edge.csv'.")


Filtered and temporally ordered data saved to 'filtered_train_3edge.csv'.


In [ ]:
#check for inside of csv (just for test, no need for run)
import pandas as pd

# Load the CSV file
file_path = "filtered_train_3edge.csv"  # Replace with your actual file path
df = pd.read_csv(file_path)

# Check if the label column contains '1'
label_column = 'Label'  # Replace with the actual label column name if different
if label_column in df.columns:
    label_distribution = df[label_column].value_counts()
    print("Label Distribution:")
    print(label_distribution)

    if 1 in label_distribution.index:
        print("The CSV contains label '1'.")
    else:
        print("The CSV does NOT contain label '1'.")
else:
    print(f"'{label_column}' column not found in the CSV.")

Label Distribution:
Label
0    1742791
Name: count, dtype: int64
The CSV does NOT contain label '1'.


# created hourly graph with 3 edges from train dataset

In [ ]:
import pandas as pd
import networkx as nx
import os
import pickle

NETWORK_FEATURES  = ['Src Port', 'Dst Port', 'Protocol']
CONTEXT_FEATURES  = [
    'Idle Min', 'Idle Max', 'Idle Mean',
    'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Subflow Fwd Byts',
    'Down/Up Ratio', 'Fwd Pkt Len Mean', 'Bwd Pkt Len Mean', 'Pkt Len Mean',
]
KNOWLEDGE_FEATURES = [
    'Fwd Header Len', 'Bwd Header Len',
    'Init Fwd Win Byts', 'Init Bwd Win Byts',
    'Fwd Seg Size Avg', 'Fwd Seg Size Min',
    'Fwd Pkt Len Max', 'Bwd Pkt Len Max',
    'Bwd Pkt Len Std', 'Pkt Len Var', 'Pkt Len Max', 'Pkt Len Std',
]


def _add_or_update_edge(G, src, dst, edge_key, label, attrs):
    """Add edge or update it; escalate label to 1 if any flow between this pair is an attack."""
    if G.has_edge(src, dst, key=edge_key):
        G[src][dst][edge_key]['label'] = max(G[src][dst][edge_key].get('label', 0), label)
        G[src][dst][edge_key].update(attrs)
    else:
        G.add_edge(src, dst, key=edge_key, label=label, **attrs)


def create_train_graphs(df, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    time_slices = [g for _, g in df.groupby(pd.Grouper(freq='h'))]

    for slice_index, slice_df in enumerate(time_slices):
        if slice_df.empty:
            continue

        print(f"Hour {slice_index}:")
        print(slice_df['Label'].value_counts())

        G = nx.MultiDiGraph()

        for _, row in slice_df.iterrows():
            src_ip = row['Src IP']
            dst_ip = row['Dst IP']

            try:
                label = int(row['Label'])
            except Exception as e:
                print(f"Skipping row due to invalid label: {row['Label']}; error: {e}")
                continue

            if pd.isna(src_ip) or pd.isna(dst_ip):
                continue

            if not G.has_node(src_ip):
                G.add_node(src_ip)
            if not G.has_node(dst_ip):
                G.add_node(dst_ip)

            _add_or_update_edge(G, src_ip, dst_ip, 'network', label,
                                {'interaction': 'network_communication',
                                 **{f: row.get(f, 0) for f in NETWORK_FEATURES}})
            _add_or_update_edge(G, src_ip, dst_ip, 'context', label,
                                {'interaction': 'context',
                                 **{f: row.get(f, 0) for f in CONTEXT_FEATURES}})
            _add_or_update_edge(G, src_ip, dst_ip, 'knowledge', label,
                                {'interaction': 'knowledge',
                                 **{f: row.get(f, 0) for f in KNOWLEDGE_FEATURES}})

        graph_path = os.path.join(output_dir, f"train_graph_hour_{slice_index}.gpickle")
        with open(graph_path, 'wb') as f:
            pickle.dump(G, f, pickle.HIGHEST_PROTOCOL)
        print(f"Train graph for hour {slice_index} saved to {graph_path}")

if __name__ == "__main__":
    df_train = pd.read_csv('filtered_train_3edge.csv')
    df_train['Timestamp'] = pd.to_datetime(df_train['Timestamp'])
    df_train = df_train.set_index('Timestamp').sort_index()
    create_train_graphs(df_train, "3ed_trai_h_graphs")


In [ ]:
import pandas as pd
import networkx as nx
import os
import pickle

NETWORK_FEATURES  = ['Src Port', 'Dst Port', 'Protocol']
CONTEXT_FEATURES  = [
    'Idle Min', 'Idle Max', 'Idle Mean',
    'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Subflow Fwd Byts',
    'Down/Up Ratio', 'Fwd Pkt Len Mean', 'Bwd Pkt Len Mean', 'Pkt Len Mean',
]
KNOWLEDGE_FEATURES = [
    'Fwd Header Len', 'Bwd Header Len',
    'Init Fwd Win Byts', 'Init Bwd Win Byts',
    'Fwd Seg Size Avg', 'Fwd Seg Size Min',
    'Fwd Pkt Len Max', 'Bwd Pkt Len Max',
    'Bwd Pkt Len Std', 'Pkt Len Var', 'Pkt Len Max', 'Pkt Len Std',
]


def _add_or_update_edge(G, src, dst, edge_key, label, attrs):
    """Add edge or update it; escalate label to 1 if any flow between this pair is an attack."""
    if G.has_edge(src, dst, key=edge_key):
        G[src][dst][edge_key]['label'] = max(G[src][dst][edge_key].get('label', 0), label)
        G[src][dst][edge_key].update(attrs)
    else:
        G.add_edge(src, dst, key=edge_key, label=label, **attrs)


def add_node_features(G):
    for node in G.nodes:
        G.nodes[node]['degree'] = G.degree[node]

    undirected_graph = nx.Graph(G)
    communities = nx.community.label_propagation_communities(undirected_graph)
    community_mapping = {
        node: cid
        for cid, community in enumerate(communities)
        for node in community
    }
    for node in G.nodes:
        cid = community_mapping.get(node, -1)
        deg = G.nodes[node]['degree']
        G.nodes[node]['community'] = cid
        G.nodes[node]['x'] = [cid, deg]

    return G


def create_train_graphs_with_node_features(df, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    time_slices = [g for _, g in df.groupby(pd.Grouper(freq='h'))]

    for slice_index, slice_df in enumerate(time_slices):
        if slice_df.empty:
            continue

        print(f"Hour {slice_index}:")
        print(slice_df['Label'].value_counts())

        G = nx.MultiDiGraph()

        for _, row in slice_df.iterrows():
            src_ip = row['Src IP']
            dst_ip = row['Dst IP']

            try:
                label = int(row['Label'])
            except Exception as e:
                print(f"Skipping row due to invalid label: {row['Label']}; error: {e}")
                continue

            if pd.isna(src_ip) or pd.isna(dst_ip):
                continue

            if not G.has_node(src_ip):
                G.add_node(src_ip)
            if not G.has_node(dst_ip):
                G.add_node(dst_ip)

            _add_or_update_edge(G, src_ip, dst_ip, 'network', label,
                                {'interaction': 'network_communication',
                                 **{f: row.get(f, 0) for f in NETWORK_FEATURES}})
            _add_or_update_edge(G, src_ip, dst_ip, 'context', label,
                                {'interaction': 'context',
                                 **{f: row.get(f, 0) for f in CONTEXT_FEATURES}})
            _add_or_update_edge(G, src_ip, dst_ip, 'knowledge', label,
                                {'interaction': 'knowledge',
                                 **{f: row.get(f, 0) for f in KNOWLEDGE_FEATURES}})

        G = add_node_features(G)

        graph_path = os.path.join(output_dir, f"train_graph_hour_{slice_index}.gpickle")
        with open(graph_path, 'wb') as f:
            pickle.dump(G, f, pickle.HIGHEST_PROTOCOL)
        print(f"Train graph for hour {slice_index} saved to {graph_path}")

if __name__ == "__main__":
    df_train = pd.read_csv('filtered_train_3edge.csv')
    df_train['Timestamp'] = pd.to_datetime(df_train['Timestamp'])
    df_train = df_train.set_index('Timestamp').sort_index()
    create_train_graphs_with_node_features(df_train, "3ed_trai_h_graphs")


# Community detection for graphs and then update the graph with the label of community for each node

In [ ]:
import networkx as nx
import os
import pickle

def detect_and_label_communities_lpa(graph):
    """
    Run LPA on the undirected projection and write community + degree into each node.
    Node feature vector x = [community_id, degree] (2-D, matches paper section 5.2).
    """
    undirected_graph = nx.Graph(graph)
    communities = nx.community.label_propagation_communities(undirected_graph)

    for community_id, community in enumerate(communities):
        for node in community:
            degree = graph.degree(node)
            graph.nodes[node]['community'] = community_id
            graph.nodes[node]['degree'] = degree
            graph.nodes[node]['x'] = [community_id, degree]

    return graph


def process_graphs_with_lpa(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    for graph_file in os.listdir(input_dir):
        if not graph_file.endswith('.gpickle'):
            continue

        graph_path = os.path.join(input_dir, graph_file)
        with open(graph_path, "rb") as f:
            G = pickle.load(f)

        G = detect_and_label_communities_lpa(G)

        updated_graph_path = os.path.join(output_dir, graph_file)
        with open(updated_graph_path, 'wb') as f:
            pickle.dump(G, f, pickle.HIGHEST_PROTOCOL)
        print(f"Updated graph saved to {updated_graph_path}")


if __name__ == "__main__":
    process_graphs_with_lpa("3ed_trai_h_graphs", "3ed_trai_h_graphs_commun")


Updated graph saved to 3ed_trai_h_graphs_commun/test_graph_hour_580.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/test_graph_hour_593.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/test_graph_hour_564.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/test_graph_hour_34.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/test_graph_hour_582.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/test_graph_hour_553.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/test_graph_hour_508.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/test_graph_hour_516.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/test_graph_hour_35.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/test_graph_hour_511.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/test_graph_hour_584.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/test_graph_hour_583.gpickle
Updated graph saved to 3ed_trai_h_graphs_commun/test_graph_hour_519.gpickle
Updated graph 

# convert Multigraph to hetrodata

In [3]:
import os
import pickle

# Path to your input directory
input_graph_dir = "3ed_trai_h_graphs_commun"

# Find the first .gpickle file in the directory
gpickle_files = [f for f in os.listdir(input_graph_dir) if f.endswith('.gpickle')]

if not gpickle_files:
    print(f"No .gpickle files found in {input_graph_dir}. Check your path!")
else:
    sample_file_path = os.path.join(input_graph_dir, gpickle_files[0])
    print(f"Inspecting file: {sample_file_path}\n" + "="*50)
    
    # Load the graph safely using pickle
    with open(sample_file_path, "rb") as f:
        G = pickle.load(f)
    
    # Tracks which features we've found for each edge type
    features_by_edge_type = {}
    
    # Scan through edges to collect their attributes
    for u, v, key, edge_attrs in G.edges(data=True, keys=True):
        if key not in features_by_edge_type:
            # Get all attribute keys, excluding 'label' since it's handled separately
            keys_found = [attr for attr in edge_attrs.keys() if attr != 'label']
            features_by_edge_type[key] = keys_found
            
    # Print the exact mapping
    for edge_type, features in features_by_edge_type.items():
        print(f"\nEdge Type: '{edge_type}' (Total Features: {len(features)})")
        print(f"Features: {features}")

Inspecting file: 3ed_trai_h_graphs_commun/train_graph_hour_2.gpickle

Edge Type: 'network' (Total Features: 4)
Features: ['interaction', 'Src Port', 'Dst Port', 'Protocol']

Edge Type: 'context' (Total Features: 11)
Features: ['interaction', 'Idle Min', 'Idle Max', 'Idle Mean', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Subflow Fwd Byts', 'Down/Up Ratio', 'Fwd Pkt Len Mean', 'Bwd Pkt Len Mean', 'Pkt Len Mean']

Edge Type: 'knowledge' (Total Features: 13)
Features: ['interaction', 'Fwd Header Len', 'Bwd Header Len', 'Init Fwd Win Byts', 'Init Bwd Win Byts', 'Fwd Seg Size Avg', 'Fwd Seg Size Min', 'Fwd Pkt Len Max', 'Bwd Pkt Len Max', 'Bwd Pkt Len Std', 'Pkt Len Var', 'Pkt Len Max', 'Pkt Len Std']


In [4]:
import torch
import os
import pickle
from torch_geometric.data import HeteroData
import networkx as nx

NETWORK_FEATURES  = ['Src Port', 'Dst Port', 'Protocol']
CONTEXT_FEATURES  = [
    'Idle Min', 'Idle Max', 'Idle Mean',
    'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Subflow Fwd Byts',
    'Down/Up Ratio', 'Fwd Pkt Len Mean', 'Bwd Pkt Len Mean', 'Pkt Len Mean',
]
KNOWLEDGE_FEATURES = [
    'Fwd Header Len', 'Bwd Header Len',
    'Init Fwd Win Byts', 'Init Bwd Win Byts',
    'Fwd Seg Size Avg', 'Fwd Seg Size Min',
    'Fwd Pkt Len Max', 'Bwd Pkt Len Max',
    'Bwd Pkt Len Std', 'Pkt Len Var', 'Pkt Len Max', 'Pkt Len Std',
]
EDGE_FEATURES = {
    'network':   NETWORK_FEATURES,
    'context':   CONTEXT_FEATURES,
    'knowledge': KNOWLEDGE_FEATURES,
}

def multiDiGraph_to_hetero_with_label(G: nx.MultiDiGraph) -> HeteroData:
    """
    Converts a MultiDiGraph to a HeteroData object.
    Node features: x = [community_id, degree]  (2-D)
    Edge features: per-type numeric feature vectors
    """
    data = HeteroData()
    node_mapping = {node: i for i, node in enumerate(G.nodes())}
    data['ip'].num_nodes = G.number_of_nodes()

    community_labels, x = [], []
    for node in G.nodes():
        cid = G.nodes[node].get('community', -1)
        deg = G.nodes[node].get('degree', G.degree(node))
        community_labels.append(cid)
        x.append([cid, deg])
    data['ip'].community = torch.tensor(community_labels, dtype=torch.long)
    data['ip'].x = torch.tensor(x, dtype=torch.float)

    for u, v, key, edge_attrs in G.edges(data=True, keys=True):
        src = node_mapping[u]
        dst = node_mapping[v]
        rel_type = ('ip', key, 'ip')
        if rel_type not in data.edge_types:
            data[rel_type].edge_index = []
            data[rel_type].edge_attr  = []
            data[rel_type].edge_label = []

        data[rel_type].edge_index.append([src, dst])
        feature_vec = [edge_attrs.get(f, 0) for f in EDGE_FEATURES.get(key, [])]
        data[rel_type].edge_attr.append(feature_vec)

        label = edge_attrs.get('label', -1)
        if label == -1:
            print(f"Warning: missing label for edge {u} -> {v} of type {key}")
        data[rel_type].edge_label.append(label)

    for rel_type in data.edge_types:
        data[rel_type].edge_index = (
            torch.tensor(data[rel_type].edge_index, dtype=torch.long).t().contiguous()
        )
        if data[rel_type].edge_attr:
            data[rel_type].edge_attr = torch.tensor(
                data[rel_type].edge_attr, dtype=torch.float
            )
        if data[rel_type].edge_label:
            data[rel_type].edge_label = torch.tensor(
                data[rel_type].edge_label, dtype=torch.long
            )
    return data


def process_and_save_hetero_graphs_with_label(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for graph_file in os.listdir(input_dir):
        if not graph_file.endswith('.gpickle'):
            continue
        graph_path = os.path.join(input_dir, graph_file)
        with open(graph_path, 'rb') as f:
            G = pickle.load(f)
        hetero_data = multiDiGraph_to_hetero_with_label(G)
        hetero_path = os.path.join(output_dir, graph_file.replace('.gpickle', '.pt'))
        torch.save(hetero_data, hetero_path)
        print(f"Saved HeteroData to {hetero_path}")


/home/rems/code/IoT-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import torch
import networkx as nx
import numpy as np
from torch_geometric.data import HeteroData


def compute_eigenvector_centrality(G: nx.MultiDiGraph) -> dict:
    """
    Compute eigenvector centrality on the undirected projection of the multigraph.
    Falls back to degree centrality if power iteration doesn't converge
    (common in sparse IoT hourly snapshots).
    """
    undirected = nx.Graph(G)  # collapse multi-edges and direction
    try:
        return nx.eigenvector_centrality(undirected, max_iter=1000, tol=1e-6)
    except nx.PowerIterationFailedConvergence:
        print("  ⚠ Eigenvector centrality did not converge — falling back to degree centrality.")
        return nx.degree_centrality(undirected)


def subsample_heterodata_by_eigenvector_centrality(
    data: HeteroData,
    G: nx.MultiDiGraph,
    sample_ratio: float = 0.5,
    seed: int = 42,
) -> HeteroData:
    """
    Subsample a HeteroData snapshot using eigenvector centrality as sampling weights.

    Since your graphs are homogeneous at the node level (all nodes are 'ip'),
    this directly maps nx node -> HeteroData 'ip' node index using the same
    node_mapping order used in multiDiGraph_to_hetero_with_label.

    Parameters:
        data        : HeteroData built by multiDiGraph_to_hetero_with_label
        G           : the original nx.MultiDiGraph (needed for centrality computation)
        sample_ratio: fraction of 'ip' nodes to keep (e.g. 0.5 = 50%)
        seed        : for reproducibility

    Returns:
        A new HeteroData with subsampled nodes and their induced edges.
    """
    np.random.seed(seed)
    torch.manual_seed(seed)

    # Step 1: compute centrality — same node order as multiDiGraph_to_hetero_with_label
    node_list = list(G.nodes())           # preserves insertion order (Python 3.7+)
    centrality = compute_eigenvector_centrality(G)

    scores = np.array([centrality.get(n, 0.0) for n in node_list], dtype=np.float64)
    total = scores.sum()
    probs = scores / total if total > 0 else np.ones(len(scores)) / len(scores)

    # Step 2: weighted sampling of node indices (no replacement)
    num_nodes = len(node_list)
    num_sample = max(1, int(num_nodes * sample_ratio))
    sampled_local = np.random.choice(num_nodes, size=num_sample, replace=False, p=probs)
    sampled_local = np.sort(sampled_local)
    kept = torch.tensor(sampled_local, dtype=torch.long)

    # Step 3: build new HeteroData
    sub = HeteroData()

    # Re-index: old local idx -> new local idx
    old_to_new = torch.full((num_nodes,), -1, dtype=torch.long)
    old_to_new[kept] = torch.arange(len(kept), dtype=torch.long)

    # Copy subsampled node features
    sub['ip'].x = data['ip'].x[kept]
    sub['ip'].community = data['ip'].community[kept]
    sub['ip'].num_nodes = len(kept)

    # Step 4: for each edge type, keep only edges between kept nodes
    # and re-index edge_index to new local ids
    for edge_type in data.edge_types:
        src_type, rel, dst_type = edge_type  # always ('ip', key, 'ip') in your case
        ei = data[edge_type].edge_index       # shape [2, num_edges]

        src_new = old_to_new[ei[0]]
        dst_new = old_to_new[ei[1]]

        # Keep edge only if both endpoints were sampled
        mask = (src_new >= 0) & (dst_new >= 0)

        sub[edge_type].edge_index = torch.stack([src_new[mask], dst_new[mask]], dim=0)

        if hasattr(data[edge_type], 'edge_attr') and data[edge_type].edge_attr is not None:
            sub[edge_type].edge_attr = data[edge_type].edge_attr[mask]

        if hasattr(data[edge_type], 'edge_label') and data[edge_type].edge_label is not None:
            sub[edge_type].edge_label = data[edge_type].edge_label[mask]

    return sub

In [6]:
def process_and_save_hetero_graphs_with_label(
    input_dir, output_dir, sample_ratio=None, seed=42
):
    """
    Converts all .gpickle graphs to HeteroData and saves as .pt.
    If sample_ratio is set (e.g. 0.5), applies eigenvector-centrality
    subsampling before saving.
    """
    os.makedirs(output_dir, exist_ok=True)

    for graph_file in sorted(os.listdir(input_dir)):
        if not graph_file.endswith('.gpickle'):
            continue

        graph_path = os.path.join(input_dir, graph_file)
        with open(graph_path, 'rb') as f:
            G = pickle.load(f)

        # Convert to HeteroData (your existing function, unchanged)
        hetero_data = multiDiGraph_to_hetero_with_label(G)

        # ── Subsampling (optional) ──────────────────────────────────────
        if sample_ratio is not None:
            print(f"  Subsampling {graph_file} at ratio={sample_ratio}...")
            hetero_data = subsample_heterodata_by_eigenvector_centrality(
                hetero_data, G, sample_ratio=sample_ratio, seed=seed
            )
        # ───────────────────────────────────────────────────────────────

        out_name = graph_file.replace('.gpickle', '.pt')
        out_path = os.path.join(output_dir, out_name)
        torch.save(hetero_data, out_path)
        print(f"  Saved → {out_path}")

In [7]:
"""# Without subsampling (original behaviour)
process_and_save_hetero_graphs_with_label(
    input_dir="3ed_trai_h_graphs_commun",
    output_dir="3ed_trai_h_graphs_hetero_graphs",
)"""

"""# With 50% eigenvector-centrality subsampling
process_and_save_hetero_graphs_with_label(
    input_dir="3ed_trai_h_graphs_commun",
    output_dir="3ed_trai_h_graphs_hetero_graphs_sub50",
    sample_ratio=0.5,
)"""

# Ablation over ratios
for ratio in [0.25, 0.5, 0.75]:
    process_and_save_hetero_graphs_with_label(
        input_dir="3ed_trai_h_graphs_commun",
        output_dir=f"3ed_trai_h_graphs_hetero_sub{int(ratio*100)}",
        sample_ratio=ratio,
    )

  Subsampling train_graph_hour_0.gpickle at ratio=0.25...
  Saved → 3ed_trai_h_graphs_hetero_sub25/train_graph_hour_0.pt
  Subsampling train_graph_hour_1.gpickle at ratio=0.25...
  Saved → 3ed_trai_h_graphs_hetero_sub25/train_graph_hour_1.pt
  Subsampling train_graph_hour_11.gpickle at ratio=0.25...
  Saved → 3ed_trai_h_graphs_hetero_sub25/train_graph_hour_11.pt
  Subsampling train_graph_hour_2.gpickle at ratio=0.25...
  Saved → 3ed_trai_h_graphs_hetero_sub25/train_graph_hour_2.pt
  Subsampling train_graph_hour_24.gpickle at ratio=0.25...
  Saved → 3ed_trai_h_graphs_hetero_sub25/train_graph_hour_24.pt
  Subsampling train_graph_hour_25.gpickle at ratio=0.25...
  Saved → 3ed_trai_h_graphs_hetero_sub25/train_graph_hour_25.pt
  Subsampling train_graph_hour_26.gpickle at ratio=0.25...
  Saved → 3ed_trai_h_graphs_hetero_sub25/train_graph_hour_26.pt
  Subsampling train_graph_hour_27.gpickle at ratio=0.25...
  Saved → 3ed_trai_h_graphs_hetero_sub25/train_graph_hour_27.pt
  Subsampling train_gr

# Test for inside of graph, no need to run it

In [ ]:
#was test for inside of .pt ( no need to run)
import torch
import os

def inspect_pt_file(file_path):
    """
    Inspects the contents of a .pt file and prints its structure.

    Parameters:
        file_path (str): Path to the .pt file.
    """
    data = torch.load(file_path)
    print(f"Inspecting file: {file_path}")
    print("-" * 40)

    # Check if it's a PyTorch Geometric HeteroData object
    if isinstance(data, dict):
        print("File contains a dictionary. Keys:")
        for key, value in data.items():
            print(f"  {key}: {type(value)}")
            if isinstance(value, torch.Tensor):
                print(f"    Tensor shape: {value.shape}")
    elif hasattr(data, 'keys') and hasattr(data, 'edge_index_dict'):
        print("File contains a HeteroData object.")
        print(f"Node types: {data.node_types}")
        for node_type in data.node_types:
            print(f"  Node type '{node_type}':")
            if 'x' in data[node_type]:
                print(f"    Node features 'x': shape {data[node_type].x.shape}")
            else:
                print("    No node features ('x') found.")
            if 'num_nodes' in data[node_type]:
                print(f"    Number of nodes: {data[node_type].num_nodes}")
        
        print(f"Edge types: {data.edge_types}")
        for edge_type in data.edge_types:
            print(f"  Edge type {edge_type}:")
            if 'edge_index' in data[edge_type]:
                print(f"    Edge index: shape {data[edge_type].edge_index.shape}")
            if 'edge_attr' in data[edge_type]:
                print(f"    Edge attributes: shape {data[edge_type].edge_attr.shape}")
    else:
        print("Unknown data format.")
    print("-" * 40)

def inspect_all_pt_files(directory):
    """
    Inspects all .pt files in a given directory.

    Parameters:
        directory (str): Path to the directory containing .pt files.
    """
    print(f"Inspecting .pt files in directory: {directory}")
    for file in os.listdir(directory):
        if file.endswith(".pt"):
            inspect_pt_file(os.path.join(directory, file))

# Directory containing your .pt files
input_graph_dir = "3ed_trai_h_graphs_hetero_graphs"

# Inspect all files in the directory
inspect_all_pt_files(input_graph_dir)


Inspecting .pt files in directory: 3ed_trai_h_graphs_hetero_graphs
Inspecting file: 3ed_trai_h_graphs_hetero_graphs/graph_hour_1876.pt
----------------------------------------
File contains a HeteroData object.
Node types: ['ip']
  Node type 'ip':
    Node features 'x': shape torch.Size([17604, 1])
    Number of nodes: 17604
Edge types: [('ip', 'network', 'ip'), ('ip', 'context', 'ip'), ('ip', 'knowledge', 'ip')]
  Edge type ('ip', 'network', 'ip'):
    Edge index: shape torch.Size([2, 19038])
    Edge attributes: shape torch.Size([19038, 10])
  Edge type ('ip', 'context', 'ip'):
    Edge index: shape torch.Size([2, 19038])
    Edge attributes: shape torch.Size([19038, 11])
  Edge type ('ip', 'knowledge', 'ip'):
    Edge index: shape torch.Size([2, 19038])
    Edge attributes: shape torch.Size([19038, 17])
----------------------------------------
Inspecting file: 3ed_trai_h_graphs_hetero_graphs/graph_hour_1967.pt
----------------------------------------
File contains a HeteroData objec

/tmp/ipykernel_1340014/2292571266.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path)


Inspecting file: 3ed_trai_h_graphs_hetero_graphs/graph_hour_1885.pt
----------------------------------------
File contains a HeteroData object.
Node types: ['ip']
  Node type 'ip':
    Node features 'x': shape torch.Size([163, 1])
    Number of nodes: 163
Edge types: [('ip', 'network', 'ip'), ('ip', 'context', 'ip'), ('ip', 'knowledge', 'ip')]
  Edge type ('ip', 'network', 'ip'):
    Edge index: shape torch.Size([2, 147])
    Edge attributes: shape torch.Size([147, 10])
  Edge type ('ip', 'context', 'ip'):
    Edge index: shape torch.Size([2, 147])
    Edge attributes: shape torch.Size([147, 11])
  Edge type ('ip', 'knowledge', 'ip'):
    Edge index: shape torch.Size([2, 147])
    Edge attributes: shape torch.Size([147, 17])
----------------------------------------
Inspecting file: 3ed_trai_h_graphs_hetero_graphs/graph_hour_1906.pt
----------------------------------------
File contains a HeteroData object.
Node types: ['ip']
  Node type 'ip':
    Node features 'x': shape torch.Size([18

In [ ]:
#was test for inside of graph ( no need to run)
import os
import networkx as nx

def inspect_community_in_gpickle(file_path):
    """
    Inspects the presence of the 'community' attribute in a .gpickle file.

    Parameters:
        file_path (str): Path to the .gpickle file.
    """
    print(f"Inspecting file: {file_path}")
    print("-" * 40)

    # Load the graph
    G = nx.read_gpickle(file_path)

    # Check for 'community' attribute in nodes
    if all('community' in G.nodes[node] for node in G.nodes()):
        print(f"All nodes have a 'community' attribute.")
        print("Sample 'community' values:")
        sample_communities = {node: G.nodes[node]['community'] for node in list(G.nodes)[:10]}
        print(sample_communities)
    else:
        missing = [node for node in G.nodes() if 'community' not in G.nodes[node]]
        print(f"Some nodes are missing the 'community' attribute. Missing nodes: {missing[:10]} (only showing first 10)")

    print(f"Total nodes: {len(G.nodes())}")
    print("-" * 40)


def inspect_all_gpickle_files(directory):
    """
    Inspects the 'community' attribute in all .gpickle files in a given directory.

    Parameters:
        directory (str): Path to the directory containing .gpickle files.
    """
    print(f"Inspecting .gpickle files in directory: {directory}")
    for file in os.listdir(directory):
        if file.endswith(".gpickle"):
            inspect_community_in_gpickle(os.path.join(directory, file))


# Directory containing your .gpickle files
input_graph_dir = "3ed_trai_h_graphs_commun"

# Inspect all files in the directory for the 'community' attribute
inspect_all_gpickle_files(input_graph_dir)


Inspecting .gpickle files in directory: 3ed_trai_h_graphs_commun
Inspecting file: 3ed_trai_h_graphs_commun/graph_hour_1886.gpickle
----------------------------------------
All nodes have a 'community' attribute.
Sample 'community' values:
{'41.24.115.254': 0, '192.168.1.190': 0, '183.69.192.168': 1, '1.1.192.168': 1, '252.229.192.168': 2, '1.190.172.17': 2, '41.72.66.76': 3, '192.168.1.32': 3, '41.184.177.116': 4, '192.168.1.31': 4}
Total nodes: 198
----------------------------------------
Inspecting file: 3ed_trai_h_graphs_commun/graph_hour_2.gpickle
----------------------------------------
All nodes have a 'community' attribute.
Sample 'community' values:
{'228.243.192.168': 0, '1.152.192.168': 0, '217.97.192.168': 0, '177.30.87.144': 1, '192.168.1.1': 1, '120.187.192.168': 2, '1.152.3.122': 2, '93.53.192.168': 0, '244.15.103.115': 3, '192.168.1.79': 3}
Total nodes: 60
----------------------------------------
Inspecting file: 3ed_trai_h_graphs_commun/graph_hour_1408.gpickle
---------